# 📘 Semaine 5 — TIMER (base de temps)

**Cours :** Microcontrôleurs STM32F103C6T6  
**Durée :** 4h30 (1h30 cours + 1h30 atelier + 1h30 homework)  
**Enseignant :** ____________________  
**Étudiant :** ____________________  
**Date :** ____________________

---

## 🎯 Objectifs pédagogiques de la semaine

À la fin de cette semaine, l'étudiant sera capable de :

1. **Décrire** la structure interne d'un timer (PSC, ARR, CNT, prescaler, auto-reload).
2. **Calculer** les valeurs de PSC et ARR pour obtenir une fréquence ou une période donnée.
3. **Configurer** un timer en mode base de temps (one-shot, périodique).
4. **Utiliser** l'interruption de débordement du timer (`HAL_TIM_PeriodElapsedCallback`).
5. **Choisir** le bon timer (TIM1 APB2, TIM2/TIM3 APB1) selon les besoins.

---

## 🗺️ Plan de la semaine

| Partie | Contenu | Durée |
|---|---|---|
| **A — Cours** | Activités 1 à 6 | 1h30 |
| **B — Atelier** | TP5 : Chenillard cadencé + chronomètre | 1h30 |
| **C — Homework** | Exercices 1 à 3 | 1h30 |
| **D — Auto-évaluation** | Checklist finale | 5 min |

---
# 🎓 PARTIE A — COURS INTÉGRÉ (1h30)

## 🔹 Activité 1 — Rappel & mise en contexte (10 min)

### 🔄 Rappel des semaines précédentes
- **GPIO** : CRL / CRH / IDR / ODR / BSRR.
- **EXTI + NVIC** : interruptions externes, priorité, préemption.
- **ISR** : courte, `volatile`, flag plutôt que traitement lourd.

### ✍️ Questions flash (2 min)
1. Que veut dire ISR ? → ...
2. Que fait `HAL_Delay()` et pourquoi est-ce déconseillé dans une ISR ? → ...
3. Quelle différence entre front montant et descendant ? → ...
4. Combien de niveaux de priorité NVIC sur STM32F103 ? → ...

### 🎯 Nouveau problème à résoudre
Comment faire clignoter une LED **exactement** toutes les 100 ms **sans bloquer** le CPU ?

---

## 🔹 Activité 2 — Structure d'un timer (25 min)

### 📖 2.1 — Schéma interne

```
                 ┌──────────────┐
   Horloge  ────▶│  Prescaler   │
   (72 MHz)      │   (PSC)      │
                 └──────┬───────┘
                        │
                        ▼
                 ┌──────────────┐
                 │  Compteur    │◀──┐
                 │   (CNT)      │   │
                 └──────┬───────┘   │
                        │            │
                        ▼            │
                 ┌──────────────┐   │
                 │  ARR compare │───┘  (reset)
                 └──────┬───────┘
                        │
                        ▼
                 ┌──────────────┐
                 │  Update IRQ  │
                 │  (UIF)       │
                 └──────────────┘
```

### 📖 2.2 — Les 3 registres clés

| Registre | Rôle | Taille |
|---|---|---|
| **PSC** | Prescaler — divise l'horloge d'entrée | 16 bits |
| **CNT** | Compteur — compte les impulsions après prescaler | 16 bits (TIM1/3) / 32 bits (TIM2) |
| **ARR** | Auto-Reload Register — valeur de débordement | 16 bits / 32 bits selon timer |

### 📖 2.3 — Formule fondamentale

```
                     F_clk
F_timer  =  ────────────────────────
             (PSC + 1) × (ARR + 1)

                     (PSC + 1) × (ARR + 1)
T_timer  =  ────────────────────────
                     F_clk
```

Où `F_clk` est l'horloge du timer :
- **TIM1** (APB2) : 72 MHz
- **TIM2 / TIM3** (APB1) : 72 MHz (car APB1 prescaler ≠ 1 → ×2)

> ⚠️ **Attention au ×2 APB1** : si APB1 prescaler ≠ 1, l'horloge du timer est **doublée**.  
> Ici : PCLK1 = 36 MHz → TIM2/TIM3 reçoivent **72 MHz**.

### 🐍 Simulation Python — Comptage d'un timer (10 min)

Visualisons l'évolution du compteur CNT avec un prescaler et une valeur ARR donnés.

In [ ]:
# ============================================================
# Simulation du comptage d'un timer STM32
# ============================================================

def simuler_timer(f_clk_hz, psc, arr, n_cycles=30):
    """Simule le compteur CNT sur n_cycles et renvoie la liste des valeurs."""
    f_timer_hz = f_clk_hz / ((psc + 1) * (arr + 1))
    periode_s  = 1 / f_timer_hz

    # On avance par pas de 1 / f_clk_hz secondes
    pas_s = 1 / f_clk_hz
    cnts = []
    cnt = 0
    evenements = []
    for i in range(int(n_cycles * (psc + 1) * (arr + 1))):
        cnt += 1
        if cnt > arr:
            cnt = 0
            evenements.append(i)
        cnts.append(cnt)
    return cnts, evenements, f_timer_hz

# Exemple : F_clk = 72 MHz, PSC = 71, ARR = 999
F_CLK = 72_000_000
PSC   = 71
ARR   = 999

cnts, evts, f_timer = simuler_timer(F_CLK, PSC, ARR, n_cycles=5)

print(f"F_clk   = {F_CLK/1e6:.0f} MHz")
print(f"PSC     = {PSC}")
print(f"ARR     = {ARR}")
print(f"F_timer = {f_timer:.6f} Hz   (période = {1/f_timer*1000:.3f} ms)")
print(f"Fréquence d'entrée du compteur : {F_CLK/(PSC+1)/1e6:.3f} MHz")
print(f"Nombre de cycles avant débordement : {ARR+1}")
print(f"Nombre d'événements 'update' sur la simu : {len(evts)}")
print(f"\n10 premières valeurs de CNT : {cnts[:10]}")
print(f"Valeurs autour du 1er débordement : {cnts[ARR-3:ARR+4]}")

---

## 🔹 Activité 3 — Les timers du STM32F103C6T6 (20 min)

### 📖 3.1 — Timers disponibles

| Timer | Bus | Type | Bits | Canaux | Usage |
|---|---|---|---|---|---|
| **TIM1** | APB2 (72 MHz) | Avancé | 16 | 4 + complémentaires | PWM moteur, encoder |
| **TIM2** | APB1 (72 MHz ×2) | Général | **32** | 4 | Base de temps, PWM |
| **TIM3** | APB1 (72 MHz ×2) | Général | 16 | 4 | Base de temps, PWM |
| **SysTick** | Cœur Cortex-M3 | Système | 24 | — | Base de temps OS (HAL) |
| **IWDG/WWDG** | APB1 | Watchdog | 12 | — | Surveillance |

### 📖 3.2 — TIM1 vs TIM2/TIM3

| Critère | TIM1 | TIM2 / TIM3 |
|---|---|---|
| Bus | APB2 | APB1 |
| Sorties complémentaires | ✅ Oui | ❌ Non |
| Dead-time | ✅ Oui | ❌ Non |
| Break input | ✅ Oui | ❌ Non |
| Compteur | 16 bits | TIM2 : 32 bits / TIM3 : 16 bits |

### 📖 3.3 — Horloge des timers APB1

Rappel de S2 :

```
SYSCLK = 72 MHz
  │
  ├── AHB /1 → HCLK = 72 MHz
  │
  ├── APB1 /2 → PCLK1 = 36 MHz
  │     │
  │     └── Si APB1 prescaler ≠ 1 → horloge timer ×2 = 72 MHz
  │
  └── APB2 /1 → PCLK2 = 72 MHz
        │
        └── Horloge timer = 72 MHz
```

> 💡 **Règle** : l'horloge des timers sur APB1 est **multipliée par 2** si le prescaler APB1 est différent de 1. Cela garantit une horloge timer à 72 MHz même si PCLK1 = 36 MHz.

---

## 🔹 Activité 4 — Modes de fonctionnement (20 min)

### 📖 4.1 — Modes de base

| Mode | Description |
|---|---|
| **One-shot** | Compte une fois puis s'arrête |
| **Périodique** | Redémarre automatiquement à chaque débordement |
| **Up-counting** | Compte de 0 à ARR |
| **Down-counting** | Compte de ARR à 0 |
| **Center-aligned** | Compte up puis down (utile pour PWM moteur) |

### 📖 4.2 — Flags et interruptions

| Flag | Signification |
|---|---|
| **UIF** | Update Interrupt Flag — débordement |
| **CC1IF..CC4IF** | Capture/Compare — utilisé en PWM |
| **TIF** | Trigger Interrupt Flag |

### 📖 4.3 — HAL : démarrer un timer

```c
HAL_TIM_Base_Start(&htim2);        // sans interruption
HAL_TIM_Base_Start_IT(&htim2);     // avec interruption
```

### 📖 4.4 — Callback HAL

```c
void HAL_TIM_PeriodElapsedCallback(TIM_HandleTypeDef *htim)
{
    if (htim->Instance == TIM2)
    {
        // Code exécuté à chaque débordement
    }
}
```

---

## 🔹 Activité 5 — Calculs PSC / ARR (15 min)

### 📖 5.1 — Méthode pas à pas

**Objectif :** obtenir une période T = 10 ms avec F_clk = 72 MHz.

**Étape 1** — Calcul du produit (PSC+1) × (ARR+1) :
```
(PSC + 1) × (ARR + 1) = F_clk × T
                     = 72 000 000 × 0.01
                     = 720 000
```

**Étape 2** — Choisir une valeur pour PSC (souvent PSC+1 = 72 ou 720 ou 7200) :
```
Si PSC + 1 = 72 :  ARR + 1 = 720 000 / 72 = 10 000
                  ARR = 9999
```

**Étape 3** — Vérifier :
```
F_timer = 72 000 000 / (72 × 10 000) = 100 Hz  ✓
T       = 10 ms                                ✓
```

### 📖 5.2 — Contraintes

- PSC : 16 bits → 0 à 65 535
- ARR : 16 bits (TIM1/3) ou 32 bits (TIM2)
- Si ARR max est insuffisant → augmenter PSC

### 🐍 Calculateur PSC/ARR (10 min)

Fonction Python pour trouver automatiquement les couples (PSC, ARR) valides.

In [ ]:
# ============================================================
# Calculateur PSC / ARR pour un timer STM32
# ============================================================

def calcul_psc_arr(f_clk_hz, periode_s, arr_max=65535, psc_max=65535, psc_pref=None):
    """
    Trouve un couple (PSC, ARR) pour obtenir une période donnée.
    - f_clk_hz    : horloge du timer en Hz
    - periode_s   : période souhaitée en secondes
    - arr_max     : 65535 (16 bits) ou 4294967295 (32 bits)
    - psc_pref    : valeur de PSC+1 préférée (ex : 72 pour 1 MHz de tick)
    Retourne (PSC, ARR) ou None.
    """
    total = f_clk_hz * periode_s   # = (PSC+1)*(ARR+1)
    if psc_pref:
        arr_plus_1 = total / psc_pref
        if arr_plus_1 != int(arr_plus_1) or arr_plus_1 - 1 > arr_max:
            return None
        return (psc_pref - 1, int(arr_plus_1) - 1)
    # Sinon, chercher un PSC raisonnable
    for psc in [1, 2, 4, 8, 16, 32, 72, 100, 720, 1000, 7200, 10000]:
        if total % psc == 0:
            arr = total // psc - 1
            if 0 <= arr <= arr_max:
                return (psc - 1, arr)
    return None

def afficher_solution(f_clk_hz, periode_s, psc, arr):
    f_timer = f_clk_hz / ((psc + 1) * (arr + 1))
    print(f"  PSC={psc:<6} ARR={arr:<10}  →  F_timer = {f_timer:.6f} Hz  (T = {1000/f_timer:.4f} ms)")

F_CLK = 72_000_000  # 72 MHz

print(f"Horloge timer : {F_CLK/1e6:.0f} MHz\n")
print(f"{'Période cible':<16}{'PSC':<8}{'ARR':<12}{'Fréq. réelle'}")
print("-" * 60)

for periode_ms in [1000, 500, 100, 10, 1, 0.1]:
    periode_s = periode_ms / 1000
    sol = calcul_psc_arr(F_CLK, periode_s, psc_pref=72)
    if sol:
        psc, arr = sol
        f_timer = F_CLK / ((psc + 1) * (arr + 1))
        print(f"{periode_ms:>6} ms       PSC={psc:<6} ARR={arr:<12} {f_timer:.4f} Hz")
    else:
        print(f"{periode_ms:>6} ms       ❌ impossible avec PSC+1=72")

### 🐍 Recherche exhaustive de configurations

In [ ]:
# Recherche toutes les combinaisons valides pour 10 ms avec 72 MHz

def toutes_solutions(f_clk_hz, periode_s, arr_max=65535, max_resultats=10):
    total = f_clk_hz * periode_s
    sols = []
    for psc_m1 in range(1, 10001):    # PSC+1
        if total % psc_m1 != 0:
            continue
        arr_m1 = total / psc_m1
        if arr_m1 - 1 > arr_max:
            continue
        if arr_m1 < 1:
            continue
        sols.append((psc_m1 - 1, int(arr_m1) - 1))
        if len(sols) >= max_resultats:
            break
    return sols

print("🔍 Toutes les configurations valides pour T = 10 ms (72 MHz)\n")
print(f"{'PSC':<8}{'ARR':<12}{'F réelle (Hz)':<18}{'Erreur'}")
print("-" * 55)
for psc, arr in toutes_solutions(72_000_000, 0.01):
    f = 72_000_000 / ((psc + 1) * (arr + 1))
    err = abs(f - 100)
    print(f"{psc:<8}{arr:<12}{f:<18.6f}{err:.2e}")

---

## 🔹 Activité 6 — QCM formatif (10 min)

**1. Le registre ARR contient :**  
A. La valeur du compteur  
B. La valeur de débordement  
C. Le prescaler  
D. La fréquence

**2. Pour F_clk = 72 MHz, PSC = 71, ARR = 999, la fréquence du timer est :**  
A. 1 Hz  
B. 100 Hz  
C. 1 kHz  
D. 10 kHz

**3. Quelle est la résolution maximale du registre PSC ?**  
A. 8 bits  
B. 12 bits  
C. 16 bits  
D. 32 bits

**4. Le flag qui signale un débordement de timer est :**  
A. CC1IF  
B. UIF  
C. TIF  
D. PR

**5. Le timer TIM2 du STM32F103C6T6 possède un compteur de :**  
A. 8 bits  
B. 16 bits  
C. 24 bits  
D. 32 bits

**6. L'horloge d'entrée de TIM2 (APB1, PCLK1 = 36 MHz, prescaler = 2) est :**  
A. 18 MHz  
B. 36 MHz  
C. 72 MHz  
D. 144 MHz

### ✅ Corrigé du QCM formatif

| Q | Réponse | Explication |
|---|---|---|
| 1 | **B — Valeur de débordement** | Arrête et reset CNT |
| 2 | **C — 1 kHz** | 72e6 / (72 × 1000) = 1000 Hz |
| 3 | **C — 16 bits** | PSC = 16 bits (0–65535) |
| 4 | **B — UIF** | Update Interrupt Flag |
| 5 | **D — 32 bits** | TIM2 est le seul timer 32 bits |
| 6 | **C — 72 MHz** | APB1 prescaler ≠ 1 → ×2 |

**Mon score : ___ / 6**

---

# 🛠️ PARTIE B — ATELIER / TP (1h30)

## 🧪 TP5 — Chenillard cadencé + chronomètre

### 🎯 Objectif
Réaliser un chenillard piloté par TIM2 avec fréquence réglable, et un chronomètre à 10 ms affiché via UART.

### 📋 Tâches à réaliser (par binôme)

| # | Tâche | Durée | Livrable |
|---|---|---|---|
| 1 | Configurer TIM2 en mode base de temps, PSC=7199, ARR=9999 → 1 Hz | 15 min | Capture CubeMX |
| 2 | Écrire `HAL_TIM_PeriodElapsedCallback` pour chenillard | 20 min | Code + démo |
| 3 | Ajouter boutons PA0/PA1/PA2 pour changer la fréquence | 20 min | Code + démo |
| 4 | Implémenter un chronomètre à 10 ms (TIM3) | 15 min | Code |
| 5 | Envoyer les secondes écoulées sur UART2 | 10 min | Démo terminal |
| 6 | Rédiger le compte-rendu | 10 min | CR |

### ⚙️ Configuration CubeMX — TIM2 à 1 Hz

**Formule :**
```
F = 72 MHz / ((PSC+1) × (ARR+1)) = 1 Hz
→ (PSC+1) × (ARR+1) = 72 000 000
→ PSC+1 = 7200,  ARR+1 = 10 000
→ PSC = 7199,     ARR = 9999
```

**Étapes dans CubeMX :**
1. *Timers → TIM2 → Clock Source = Internal Clock*
2. *Parameter Settings* :
   - Prescaler (PSC) : `7199`
   - Counter Period (ARR) : `9999`
   - Counter Mode : Up
3. *NVIC Settings* : cocher **TIM2 global interrupt**
4. Priorité : Preemption 1, Sub 0
5. Générer le code

In [ ]:
/* ============================================================
   TP5 - Chenillard cadencé + chronomètre
   - TIM2 : base de temps 1 Hz pour le chenillard
   - TIM3 : chronomètre à 100 Hz (10 ms)
   - Boutons PA0/PA1/PA2 : changement de fréquence
   - LEDs : PC13, PC14, PC15, PA1 (chenillard)
   ============================================================ */

#include "main.h"

TIM_HandleTypeDef htim2;
TIM_HandleTypeDef htim3;

volatile uint32_t ticks_10ms   = 0;   // compteur 10 ms
volatile uint8_t  vitesse      = 0;   // 0 = 1Hz, 1 = 2Hz, 2 = 4Hz
volatile uint8_t  etape        = 0;   // position chenillard

/* --- LEDs utilisées pour le chenillard --- */
static const uint16_t leds[4] = {
    GPIO_PIN_13,  // PC13
    GPIO_PIN_14,  // PC14
    GPIO_PIN_15,  // PC15
    GPIO_PIN_1,   // PA1
};

/* --- Affiche l'étape courante sur les LEDs --- */
static void afficher_etape(uint8_t e)
{
    HAL_GPIO_WritePin(GPIOC, GPIO_PIN_13, GPIO_PIN_RESET);
    HAL_GPIO_WritePin(GPIOC, GPIO_PIN_14, GPIO_PIN_RESET);
    HAL_GPIO_WritePin(GPIOC, GPIO_PIN_15, GPIO_PIN_RESET);
    HAL_GPIO_WritePin(GPIOA, GPIO_PIN_1,  GPIO_PIN_RESET);

    switch (e)
    {
        case 0: HAL_GPIO_WritePin(GPIOC, GPIO_PIN_13, GPIO_PIN_SET); break;
        case 1: HAL_GPIO_WritePin(GPIOC, GPIO_PIN_14, GPIO_PIN_SET); break;
        case 2: HAL_GPIO_WritePin(GPIOC, GPIO_PIN_15, GPIO_PIN_SET); break;
        case 3: HAL_GPIO_WritePin(GPIOA, GPIO_PIN_1,  GPIO_PIN_SET); break;
    }
}

/* --- Callback timer : appelé à chaque UIF --- */
void HAL_TIM_PeriodElapsedCallback(TIM_HandleTypeDef *htim)
{
    if (htim->Instance == TIM3)
    {
        ticks_10ms++;   // chronomètre à 10 ms
    }
    else if (htim->Instance == TIM2)
    {
        // Le chenillard avance seulement si la vitesse le permet
        static uint8_t diviseur = 0;
        uint8_t cible = (vitesse == 0) ? 1 : (vitesse == 1 ? 2 : 4);
        diviseur = (diviseur + 1) % cible;
        if (diviseur == 0)
        {
            etape = (etape + 1) & 0x03;
            afficher_etape(etape);
        }
    }
}

int main(void)
{
    HAL_Init();
    SystemClock_Config();
    MX_GPIO_Init();
    MX_TIM2_Init();
    MX_TIM3_Init();
    MX_USART2_UART_Init();

    HAL_TIM_Base_Start_IT(&htim2);
    HAL_TIM_Base_Start_IT(&htim3);

    afficher_etape(0);

    uint32_t dernier_affichage = 0;
    char buf[32];

    while (1)
    {
        // Affiche les secondes toutes les 100 × 10 ms = 1 s
        if (ticks_10ms - dernier_affichage >= 100)
        {
            dernier_affichage = ticks_10ms;
            int sec = ticks_10ms / 100;
            int cs  = (ticks_10ms % 100) / 10;   // centièmes
            int snprintf_ret = snprintf(buf, sizeof(buf), "Temps : %d.%02d s\r\n", sec, cs);
            HAL_UART_Transmit(&huart2, (uint8_t*)buf, snprintf_ret, 100);
        }

        // Gestion des boutons de vitesse
        if (HAL_GPIO_ReadPin(GPIOA, GPIO_PIN_0) == GPIO_PIN_SET)
        {
            vitesse = 0;  // 1 Hz
            HAL_Delay(200);
        }
        else if (HAL_GPIO_ReadPin(GPIOA, GPIO_PIN_2) == GPIO_PIN_SET)
        {
            vitesse = 2;  // 4 Hz
            HAL_Delay(200);
        }
    }
}

### 🔍 Analyse du code

| Élément | Rôle |
|---|---|
| `HAL_TIM_Base_Start_IT(&htim2)` | Démarre TIM2 avec interruption |
| `HAL_TIM_PeriodElapsedCallback` | Callback appelé à chaque UIF |
| `volatile` | Variables modifiées par ISR |
| `ticks_10ms` | Compteur 10 ms (chronomètre) |
| `diviseur` | Permet de diviser la fréquence du chenillard |

### 📖 Configuration TIM3 (chronomètre 10 ms)

```
F = 72 MHz / ((PSC+1) × (ARR+1)) = 100 Hz  →  T = 10 ms
→ (PSC+1) × (ARR+1) = 720 000
→ PSC = 719, ARR = 999
```

### 🐍 Simulation Python — Chenillard cadencé (15 min)

Simulons le fonctionnement du chenillard avec changement de vitesse.

In [ ]:
# ============================================================
# Simulation du chenillard avec TIM2
# ============================================================

def simuler_chenillard(duree_s, vitesse_hz, n_leds=4):
    """
    Simule l'état du chenillard à chaque pas de 10 ms.
    Retourne une liste de tuples (t_ms, led_allumee).
    """
    pas_ms     = 10
    periode_ms = 1000 / vitesse_hz
    resultats  = []
    etape      = 0
    prochain   = periode_ms
    for t_ms in range(0, int(duree_s * 1000), pas_ms):
        if t_ms >= prochain:
            etape = (etape + 1) % n_leds
            prochain += periode_ms
        resultats.append((t_ms, etape))
    return resultats

# Test : chenillard 1 Hz pendant 4 secondes
print("📺 Chenillard 1 Hz (4 s)\n")
for t, led in simuler_chenillard(4.0, vitesse_hz=1):
    if t % 100 == 0:  # afficher toutes les 100 ms
        barre = "▁" * led + "█" + "▁" * (3 - led)
        print(f"t = {t:>4} ms  LED{led}  {barre}")

print("\n📺 Chenillard 4 Hz (2 s)\n")
for t, led in simuler_chenillard(2.0, vitesse_hz=4):
    if t % 50 == 0:  # afficher toutes les 50 ms
        barre = "▁" * led + "█" + "▁" * (3 - led)
        print(f"t = {t:>4} ms  LED{led}  {barre}")

### 🐍 Simulation Python — Chronomètre (15 min)

Comparons un chronomètre logiciel basé sur un timer matériel.

In [ ]:
# ============================================================
# Chronomètre à 10 ms (TIM3) — simulation du formatage
# ============================================================

def formater_chrono(ticks_10ms):
    """
    Formate un nombre de ticks (10 ms) en MM:SS.cc
    Ex : 1234 → '00:12.34'
    """
    total_cs = ticks_10ms              # centièmes de seconde
    minutes  = total_cs // 6000
    secondes = (total_cs // 100) % 60
    centiemes = total_cs % 100
    return f"{minutes:02d}:{secondes:02d}.{centiemes:02d}"

print("⏱️  Formatage du chronomètre\n")
print(f"{'Ticks (10 ms)':<18}{'Temps affiché'}")
print("-" * 40)
for ticks in [0, 5, 99, 100, 250, 1234, 5999, 6000, 60000]:
    print(f"{ticks:<18}{formater_chrono(ticks)}")

# Simulation d'une seconde entière
print("\n⏱️  Déroulement sur 1 seconde (extrait tous les 100 ms) :")
for t_ms in range(0, 1050, 100):
    ticks = t_ms // 10
    print(f"  Après {t_ms:>4} ms  →  {formater_chrono(ticks)}")

### 📝 Compte-rendu de TP5

**Nom :** __________________  **Prénom :** __________________  **Binôme :** __________________

**1. Configuration CubeMX**
- TIM2 : PSC = ... , ARR = ... , Mode = ...
- TIM3 : PSC = ... , ARR = ... , Mode = ...
- Fréquence TIM2 : ... Hz (soit T = ... ms)
- Fréquence TIM3 : ... Hz (soit T = ... ms)
- Priorités NVIC : TIM2 = ... , TIM3 = ...

**2. Code ajouté dans `main.c`**
```c
// Colle ici ton code
```

**3. Observation du chenillard**
- Fréquence 1 Hz : OK / KO
- Changement de vitesse : OK / KO
- Problèmes observés : ...

**4. Chronomètre**
- Le chrono est-il précis à 10 ms ? ...
- Comparer avec un chrono du téléphone sur 30 s : écart = ... s
- Cause des écarts éventuels : ...

**5. Mesures à l'oscilloscope**
- Fréquence mesurée de TIM2 (via toggle sur une pin) : ... Hz
- Fréquence attendue : ... Hz
- Écart relatif : ... %

**6. Problèmes rencontrés**
- ...

**7. Solutions apportées**
- ...

### 🧪 Exercice bonus — Mesurer le temps d'exécution d'une fonction

Utiliser TIM2 (à 1 MHz, soit 1 µs par tick) pour mesurer le temps d'exécution d'une fonction.

**Cahier des charges :**
- Configurer TIM2 : PSC=71, ARR=65535 → tick = 1 µs
- Démarrer TIM2 en mode one-shot
- Mesurer le temps d'exécution d'une boucle `for` de 1000 itérations
- Afficher le résultat sur UART2

In [ ]:
// Squelette solution bonus

// TIM2 configuré à 1 MHz (PSC=71, ARR=65535)
HAL_TIM_Base_Start(&htim2);
__HAL_TIM_SET_COUNTER(&htim2, 0);   // remise à zéro CNT

// Code à mesurer
volatile uint32_t somme = 0;
for (uint32_t i = 0; i < 1000; i++)
{
    somme += i;
}

uint32_t duree_us = __HAL_TIM_GET_COUNTER(&htim2);
HAL_TIM_Base_Stop(&htim2);

char buf[64];
int n = snprintf(buf, sizeof(buf), "Duree = %lu us\r\n", duree_us);
HAL_UART_Transmit(&huart2, (uint8_t*)buf, n, 100);

---

# 🏠 PARTIE C — HOMEWORK (1h30)

## 📚 Exercices à rendre

### 🧩 Exercice 1 — Calculs PSC / ARR (30 min)

Pour un timer TIM2 à **72 MHz**, calculer les couples (PSC, ARR) pour obtenir les fréquences suivantes.

| # | Fréquence cible | Période | PSC | ARR |
|---|---|---|---|---|
| 1 | 1 Hz | 1 s | ? | ? |
| 2 | 10 Hz | 100 ms | ? | ? |
| 3 | 100 Hz | 10 ms | ? | ? |
| 4 | 1 kHz | 1 ms | ? | ? |
| 5 | 10 kHz | 100 µs | ? | ? |
| 6 | 50 Hz (servo) | 20 ms | ? | ? |
| 7 | 1 MHz | 1 µs | ? | ? |
| 8 | 0.5 Hz | 2 s | ? | ? |
| 9 | 250 Hz | 4 ms | ? | ? |
| 10 | 2.5 kHz | 400 µs | ? | ? |

👉 Utilise le calculateur Python ci-dessus pour vérifier.

In [ ]:
# Corrigé Exercice 1

cibles = [
    (1,       "1 Hz"),
    (10,      "10 Hz"),
    (100,     "100 Hz"),
    (1_000,   "1 kHz"),
    (10_000,  "10 kHz"),
    (50,      "50 Hz"),
    (1_000_000, "1 MHz"),
    (0.5,     "0.5 Hz"),
    (250,     "250 Hz"),
    (2_500,   "2.5 kHz"),
]

F_CLK = 72_000_000

print(f"{'#':<4}{'Cible':<12}{'PSC':<8}{'ARR':<12}{'F réelle'}")
print("-" * 55)
for i, (f, nom) in enumerate(cibles, 1):
    T = 1 / f
    # On privilégie un PSC qui laisse ARR ≤ 65535
    total = F_CLK * T
    # chercher le plus petit PSC+1 tel que ARR+1 ≤ 65536 et divisible
    sol = None
    for psc in range(1, 65536):
        if total % psc == 0:
            arr = total / psc - 1
            if 0 <= arr <= 65535:
                sol = (psc - 1, int(arr))
                break
    if sol:
        psc_v, arr_v = sol
        f_reelle = F_CLK / ((psc_v + 1) * (arr_v + 1))
        print(f"{i:<4}{nom:<12}PSC={psc_v:<4} ARR={arr_v:<10} {f_reelle:.4f} Hz")
    else:
        print(f"{i:<4}{nom:<12}❌ impossible")

### 🧩 Exercice 2 — Analyse d'un callback (30 min)

Analyser le code suivant et répondre aux questions :

```c
volatile uint32_t cnt = 0;

void HAL_TIM_PeriodElapsedCallback(TIM_HandleTypeDef *htim)
{
    if (htim->Instance == TIM2)
    {
        cnt++;
        if (cnt % 10 == 0)
            HAL_GPIO_TogglePin(GPIOC, GPIO_PIN_13);
    }
}
```

**Hypothèse :** TIM2 configuré à 100 Hz.

**Questions :**
1. Quelle est la fréquence de clignotement de la LED PC13 ?
2. Quelle est la période de clignotement ?
3. Combien de fois le callback est-il appelé par seconde ?
4. Combien de fois la LED commute-t-elle par seconde ?
5. Si on changeait TIM2 à 1 kHz, quelle serait la nouvelle fréquence de clignotement ?

In [ ]:
# Corrigé Exercice 2

def frequence_led(f_timer_hz, diviseur):
    """Fréquence de clignotement de la LED."""
    # Le callback est appelé f_timer fois par seconde.
    # La LED commute chaque fois que cnt % diviseur == 0.
    # Un cycle complet = 2 commutations → 1 allumage + 1 extinction.
    nb_commutations_s = f_timer_hz / diviseur
    return nb_commutations_s / 2   # fréquence du cycle complet

for f_timer, div in [(100, 10), (1000, 10), (100, 1), (1000, 100)]:
    f = frequence_led(f_timer, div)
    T = 1 / f if f > 0 else float('inf')
    print(f"TIM2 à {f_timer:>4} Hz, diviseur={div:<4} → LED à {f:.4f} Hz (T = {T*1000:.1f} ms)")

### 🧩 Exercice 3 — Lecture du RM0008 (30 min)

Lire le **chapitre 15 (TIM)** du RM0008 et répondre :

1. Combien de bits possède le registre **PSC** ? **ARR** ? **CNT** (pour TIM1/TIM3) ?
2. Quelle est la différence entre **up-counting** et **down-counting** ?
3. Que fait le bit **URS** (Update Request Source) dans `TIMx_CR1` ?
4. Que fait le bit **OPM** (One Pulse Mode) ?
5. Comment vérifier que le timer a bien débordé sans utiliser l'interruption ?
6. Que se passe-t-il si on écrit une nouvelle valeur dans ARR pendant que le timer tourne ?

### ✍️ Réponses — Exercice 3

1. ...
2. ...
3. ...
4. ...
5. ...
6. ...

---

## 🧮 Exercice supplémentaire — Chronomètre de haute précision (optionnel)

Simule en Python un chronomètre basé sur un timer 1 µs (72 MHz / PSC=71) et calcule la dérive sur 1 heure en supposant une imprécision de ±20 ppm sur le quartz.

In [ ]:
# ============================================================
# Dérive d'un chronomètre basé sur un quartz 8 MHz
# ============================================================

def derive_chrono(duree_s, ppm):
    """Dérive en secondes après `duree_s` avec une précision de `ppm`."""
    return duree_s * ppm / 1e6

print("⏱️  Dérive d'un quartz 8 MHz (±20 ppm typique)\n")
print(f"{'Durée':<14}{'Dérive (±ppm)':<18}{'Dérive (s)'}")
print("-" * 50)
for duree_s, label in [(1, "1 s"), (60, "1 min"), (3600, "1 h"), (86400, "24 h")]:
    for ppm in [20]:
        d = derive_chrono(duree_s, ppm)
        print(f"{label:<14}{ppm:<18}{d*1000:.3f} ms")

# Avec un quartz de meilleure qualité (TCXO ±1 ppm)
print("\n⏱️  Avec TCXO (±1 ppm) :\n")
for duree_s, label in [(1, "1 s"), (60, "1 min"), (3600, "1 h"), (86400, "24 h")]:
    d = derive_chrono(duree_s, 1)
    print(f"  {label:<8} → {d*1000:.3f} ms")

---
# ✅ PARTIE D — AUTO-ÉVALUATION Semaine 5

Coche ce que tu maîtrises.

- [ ] Je connais la structure d'un timer (PSC, ARR, CNT).
- [ ] Je sais écrire la formule F_timer = F_clk / ((PSC+1)×(ARR+1)).
- [ ] Je connais les timers disponibles sur STM32F103C6T6 (TIM1, TIM2, TIM3).
- [ ] Je comprends le ×2 APB1 sur les timers.
- [ ] Je sais configurer un timer en CubeMX.
- [ ] Je sais démarrer un timer avec `HAL_TIM_Base_Start_IT()`.
- [ ] Je sais écrire `HAL_TIM_PeriodElapsedCallback`.
- [ ] Je sais utiliser `volatile` pour les variables partagées avec l'ISR.
- [ ] J'ai implémenté un chenillard cadencé.
- [ ] J'ai implémenté un chronomètre à 10 ms.
- [ ] J'ai mesuré la fréquence réelle avec un oscilloscope.
- [ ] J'ai rédigé mon compte-rendu de TP5.
- [ ] J'ai complété le tableau PSC/ARR.
- [ ] J'ai lu le chapitre 15 du RM0008.

### 📊 Mon score : ___ / 14

| Score | Interprétation |
|---|---|
| 12–14 | ✅ Prêt pour la S6 (PWM) |
| 8–11 | ⚠️ Revoir les points manquants |
| < 8 | 🔁 Reprendre les activités 2 à 5 |

---
# 📚 RESSOURCES Semaine 5

### Documents officiels
- 📄 **RM0008** — chapitre 15 (Advanced-control timers), chapitre 16 (General-purpose timers)
- 📄 **Datasheet STM32F103x6** — section 2.3.11 (Timers)
- 📄 **UM1850** — HAL TIM documentation

### Outils
- **STM32CubeMX** — onglet *Timers* → TIM1 / TIM2 / TIM3
- **Oscilloscope** ou **analyseur logique** pour mesurer la fréquence réelle
- **STM32CubeProgrammer** pour lire PSC/ARR/CNT en live

### Vidéos
- *STM32 TIMER Tutorial (HAL)* — ControllersTech
- *Understanding STM32 Timers* — YouTube

### Bonnes pratiques
- Calculer PSC/ARR avec la formule avant de configurer
- Utiliser `volatile` pour toute variable partagée avec l'ISR
- ISR de timer : courte, pas de `HAL_Delay`, pas de `printf`
- Préférer `HAL_TIM_Base_Start_IT()` pour les timers périodiques

---

### 🔗 Passage à la semaine 6

**Prochaine séance :** PWM — Pulse Width Modulation  
- Génération d'un signal PWM (fréquence + duty cycle)
- Canaux de comparaison (CCR1..CCR4)
- Modes : edge-aligned, center-aligned
- Applications : LED (dimming), servo moteur, moteur DC

**Préparation :** Lire la section PWM du chapitre 15 du RM0008.

---

**Fin du notebook — Semaine 5** ✨